#  Feature Engineering — Transilien Challenge

## I. Importation des bibliothèques

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print(" Bibliothèques importées avec succès !")

 Bibliothèques importées avec succès !


## II. Chargement des données

In [3]:
x_train = pd.read_csv('../data/raw/x_train.csv',index_col=0)
y_train = pd.read_csv('../data/raw/y_train.csv',index_col=0)
x_test = pd.read_csv('../data/raw/x_test.csv',index_col=0)
print("Données chargées avec succès !")
print(f"x_train : {x_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"x_test  : {x_test.shape}") 

Données chargées avec succès !
x_train : (667264, 11)
y_train : (667264, 1)
x_test  : (20657, 10)


## III. Feature Engineering
### 1. Copies indépendantes des données

In [4]:
# Copies indépendantes pour ne pas modifier les données originales
x_train = x_train.copy()
x_test  = x_test.copy()
y_train_clipped = y_train.copy()

print("Copies effectuées !")

Copies effectuées !


### 2. Features temporelles + Clipping

In [5]:
lag_cols = ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']

# Features temporelles
for df in [x_train, x_test]:
    df['date']         = pd.to_datetime(df['date'])
    df['jour_semaine'] = df['date'].dt.dayofweek
    df['mois']         = df['date'].dt.month
    df['jour_mois']    = df['date'].dt.day

# Clipping
y_train_clipped['p0q0']   = y_train_clipped['p0q0'].clip(-4, 4)
x_train[lag_cols] = x_train[lag_cols].clip(-15, 15)
x_test[lag_cols]  = x_test[lag_cols].clip(-15, 15)
print(f"\ny_train → Min : {y_train_clipped['p0q0'].min()}  Max : {y_train_clipped['p0q0'].max()}")
print(f"x_train lags → Min : {x_train[lag_cols].min().min()}  Max : {x_train[lag_cols].max().max()}")


y_train → Min : -4.0  Max : 4.0
x_train lags → Min : -15.0  Max : 15.0


### 3. Features statistiques par gare et par train

In [6]:
# DataFrame temporaire
df_train = x_train.copy()
df_train['p0q0'] = y_train_clipped['p0q0'].values

# Stats par gare
gare_stats = df_train.groupby('gare')['p0q0'].agg(
    gare_mean='mean',
    gare_std='std',
    gare_median='median'
).reset_index()

# Stats par train
train_stats = df_train.groupby('train')['p0q0'].agg(
    train_mean='mean',
    train_std='std',
    train_median='median'
).reset_index()

# Fusion séparée pour éviter les _x et _y
x_train = x_train.merge(gare_stats, on='gare', how='left')
x_test  = x_test.merge(gare_stats, on='gare', how='left')

x_train = x_train.merge(train_stats, on='train', how='left')
x_test  = x_test.merge(train_stats, on='train', how='left')

# Remplacement des NaN
global_mean = df_train['p0q0'].mean()
x_train = x_train.fillna(global_mean)
x_test  = x_test.fillna(global_mean)

print("Features statistiques ajoutées !")
print(f"Valeurs manquantes x_train : {x_train.isnull().sum().sum()}")
print(f"Valeurs manquantes x_test  : {x_test.isnull().sum().sum()}")
print(f"\nx_train shape : {x_train.shape}")
print(f"x_test  shape : {x_test.shape}")
print(f"\nColonnes : {x_train.columns.tolist()}")

Features statistiques ajoutées !
Valeurs manquantes x_train : 0
Valeurs manquantes x_test  : 0

x_train shape : (667264, 20)
x_test  shape : (20657, 19)

Colonnes : ['Unnamed: 0', 'train', 'gare', 'date', 'arret', 'p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4', 'jour_semaine', 'mois', 'jour_mois', 'gare_mean', 'gare_std', 'gare_median', 'train_mean', 'train_std', 'train_median']


### 4. Suppression des colonnes inutiles

In [7]:
# Colonnes à supprimer (texte + index parasite)
cols_to_drop = ['train', 'gare', 'date', 'Unnamed: 0']

# Suppression
x_train = x_train.drop(columns=cols_to_drop, errors='ignore')
x_test  = x_test.drop(columns=cols_to_drop, errors='ignore')

print("Colonnes supprimées !")
print(f"\nColonnes finales :")
for col in x_train.columns.tolist():
    print(f"  - {col}")
print(f"\nx_train shape : {x_train.shape}")
print(f"x_test  shape : {x_test.shape}")

Colonnes supprimées !

Colonnes finales :
  - arret
  - p2q0
  - p3q0
  - p4q0
  - p0q2
  - p0q3
  - p0q4
  - jour_semaine
  - mois
  - jour_mois
  - gare_mean
  - gare_std
  - gare_median
  - train_mean
  - train_std
  - train_median

x_train shape : (667264, 16)
x_test  shape : (20657, 16)


### 5. Features supplémentaires

In [8]:
# 1. Tendance du retard
x_train['trend_q'] = x_train['p0q2'] - x_train['p0q3']
x_train['trend_p'] = x_train['p2q0'] - x_train['p3q0']
x_test['trend_q']  = x_test['p0q2'] - x_test['p0q3']
x_test['trend_p']  = x_test['p2q0'] - x_test['p3q0']

# 2. Moyenne des lags
x_train['mean_lag_q'] = x_train[['p0q2','p0q3','p0q4']].mean(axis=1)
x_train['mean_lag_p'] = x_train[['p2q0','p3q0','p4q0']].mean(axis=1)
x_test['mean_lag_q']  = x_test[['p0q2','p0q3','p0q4']].mean(axis=1)
x_test['mean_lag_p']  = x_test[['p2q0','p3q0','p4q0']].mean(axis=1)

# 3. Valeur absolue des lags
x_train['abs_p0q2'] = x_train['p0q2'].abs()
x_train['abs_p2q0'] = x_train['p2q0'].abs()
x_test['abs_p0q2']  = x_test['p0q2'].abs()
x_test['abs_p2q0']  = x_test['p2q0'].abs()

# 4. Stats par arrêt (map au lieu de merge)
arret_stats = df_train.groupby('arret')['p0q0'].agg(
    arret_mean='mean',
    arret_std='std'
).reset_index()

arret_mean_map = arret_stats.set_index('arret')['arret_mean']
arret_std_map  = arret_stats.set_index('arret')['arret_std']

x_train['arret_mean'] = x_train['arret'].map(arret_mean_map)
x_train['arret_std']  = x_train['arret'].map(arret_std_map)
x_test['arret_mean']  = x_test['arret'].map(arret_mean_map)
x_test['arret_std']   = x_test['arret'].map(arret_std_map)

print("✅ Features supplémentaires ajoutées !")
print(f"\nx_train shape : {x_train.shape}")
print(f"x_test  shape : {x_test.shape}")
print(f"\nColonnes finales :")
for col in x_train.columns.tolist():
    print(f"  - {col}")

✅ Features supplémentaires ajoutées !

x_train shape : (667264, 24)
x_test  shape : (20657, 24)

Colonnes finales :
  - arret
  - p2q0
  - p3q0
  - p4q0
  - p0q2
  - p0q3
  - p0q4
  - jour_semaine
  - mois
  - jour_mois
  - gare_mean
  - gare_std
  - gare_median
  - train_mean
  - train_std
  - train_median
  - trend_q
  - trend_p
  - mean_lag_q
  - mean_lag_p
  - abs_p0q2
  - abs_p2q0
  - arret_mean
  - arret_std


### 6. Sauvegarde des données traitées

In [9]:
# Sauvegarde des données traitées
x_train.to_csv('../data/processed/x_train_processed.csv')
x_test.to_csv('../data/processed/x_test_processed.csv')
y_train_clipped.to_csv('../data/processed/y_train_processed.csv')

print("Données sauvegardées dans data/processed/ !")
print(f"\nx_train_processed : {x_train.shape}")
print(f"x_test_processed  : {x_test.shape}")
print(f"y_train_processed : {y_train_clipped.shape}")

Données sauvegardées dans data/processed/ !

x_train_processed : (667264, 24)
x_test_processed  : (20657, 24)
y_train_processed : (667264, 1)
